In [117]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error

In [119]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [121]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [123]:
tahminler = []
ts = TimeSeriesSplit(n_splits=2)

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        train_data = df_target[df_target['yıl'] <= 2021]
        val_data = df_target[df_target['yıl'] == 2022]
        test_data = df_target[df_target['yıl'] == 2023]

        if val_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = train_data[hedef]
        X_val = val_data[features]
        y_val = val_data[hedef]
        X_test = test_data[features]
        y_test = test_data[hedef]

        mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

        if len(X_train) < 10 or len(set(y_train)) <= 1:
            continue

        # XGBoost + RandomizedSearchCV (genis parametre araligi)
        xgb_params = {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "min_child_weight": [1, 3],
            "gamma": [0, 0.1, 0.2],
            "reg_alpha": [0, 0.1],
            "reg_lambda": [1, 2]
        }

        try:
            grid = RandomizedSearchCV(
                estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
                param_distributions=xgb_params,
                scoring=mae_scorer,
                n_iter=50,
                cv=ts,
                n_jobs=-1,
                verbose=0,
                random_state=42
            )
            grid.fit(X_train, y_train)
            xgb = grid.best_estimator_
        except:
            xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)

        # Tahminler
        y_val_pred_xgb = xgb.predict(X_val)[0]
        y_test_pred_xgb = xgb.predict(X_test)[0]

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2022": y_val.values[0],
            "tahmin_2022_xgb": y_val_pred_xgb,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_xgb": y_test_pred_xgb,
        })

In [125]:
df_tahmin = pd.DataFrame(tahminler)

In [127]:
# Tahmin DataFrame
df_tahmin = pd.DataFrame(tahminler)

# Doğruluk (%)
df_tahmin["xgb_yuzde_dog"] = 100 - (
    (np.abs(df_tahmin["gercek_2023"] - df_tahmin["tahmin_2023_xgb"]) / df_tahmin["gercek_2023"]) * 100
)
df_tahmin["xgb_yuzde_dog"] = df_tahmin["xgb_yuzde_dog"].clip(0, 100)

# Ortalamayı yazdır
print("\n📊 Ortalama XGBoost Doğruluk (2023):")
print(f"XGB doğruluk ort.: {df_tahmin['xgb_yuzde_dog'].mean():.2f}%")

# Kaydet
df_tahmin.to_csv("sehir_bazli_2023_tahmin_xgb.csv", index=False)


📊 Ortalama XGBoost Doğruluk (2023):
XGB doğruluk ort.: 79.89%


In [129]:
# Performans metrikleri
mae = mean_absolute_error(df_tahmin["gercek_2023"], df_tahmin["tahmin_2023_xgb"])
rmse = mean_squared_error(df_tahmin["gercek_2023"], df_tahmin["tahmin_2023_xgb"], squared=False)

print("\n📉 XGBoost Performans Metrikleri (2023):")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")


📉 XGBoost Performans Metrikleri (2023):
MAE  : 248.50
RMSE : 880.46


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [131]:
# Bölge haritası
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgesel toplamlar
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[
    ["gercek_2023", "tahmin_2023_xgb"]
].sum().reset_index()

bolgesel.to_csv("bolge_bazli_2023_tahmin_xgb.csv", index=False)
print("\n✅ XGBoost tahminleri CSV olarak kaydedildi.")


✅ XGBoost tahminleri CSV olarak kaydedildi.


In [133]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_xgb.csv")  # Gerekirse dosya adı değiştir

# 2. Doğruluk hesapla (sadece XGBoost için)
df["xgb_yuzde_dog"] = 100 - (
    np.abs(df["gercek_2023"] - df["tahmin_2023_xgb"]) / df["gercek_2023"] * 100
)
df["xgb_yuzde_dog"] = df["xgb_yuzde_dog"].clip(0, 100)

# 3. Ortalama doğruluğu yazdır
ortalama = round(df["xgb_yuzde_dog"].mean(), 2)
print("\nXGBoost Doğruluk Ortalaması (2023):")
print(f" - XGB: {ortalama}%")

# 4. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler_xgb.csv", index=False)
print("\n✅ Sonuçlar 'sehir_tahmin_yuzdeler_xgb.csv' olarak kaydedildi.")


XGBoost Doğruluk Ortalaması (2023):
 - XGB: 79.89%

✅ Sonuçlar 'sehir_tahmin_yuzdeler_xgb.csv' olarak kaydedildi.


In [135]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin_xgb.csv")  # Dosya adını xgb'ye göre güncelledik

# 2. Yüzdelik doğruluk hesapla (XGBoost için)
df["xgb_yuzde_dog"] = 100 - (
    np.abs(df["gercek_2023"] - df["tahmin_2023_xgb"]) / df["gercek_2023"] * 100
)
df["xgb_yuzde_dog"] = df["xgb_yuzde_dog"].clip(0, 100)

# 3. Ortalama doğruluğu hesapla ve yazdır
ortalama = round(df["xgb_yuzde_dog"].mean(), 2)
print("\nXGBoost Bölge Bazlı Doğruluk Ortalaması (2023):")
print(f" - XGB: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler_xgb.csv", index=False)
print("\n✅ Sonuçlar 'bolge_tahmin_yuzdeler_xgb.csv' olarak kaydedildi.")


XGBoost Bölge Bazlı Doğruluk Ortalaması (2023):
 - XGB: 87.42%

✅ Sonuçlar 'bolge_tahmin_yuzdeler_xgb.csv' olarak kaydedildi.
